## Sptial statistics MLE

In [2]:
import numpy as np
from scipy.optimize import minimize
from scipy.spatial.distance import cdist
import matplotlib.pyplot as plt
import time

In [15]:
# =======================================================
# 1. 데이터 생성
# =======================================================

def generate_data(n=100, true_mu = 5.0, true_a = 2.0, true_b = 3.0):
    S = np.random.uniform(0, 10, size=(n, 2)) # create size n x 2 matrixh
    D = cdist(S, S, metric='euclidean') # Compute distance between each pair of the two collections of inputs
    
    # Assumption 1 cov(Y_i ,Y_j) = a * exp(-||S_i - S_j|| / b)
    Sigma = true_a * np.exp(-D / true_b) + 1e-8 * np.eye(n) # eye --> return diag = 1 and zeros elsewhere ==> identity matrix
    
    # Assumption 2 Y = (Y_1, ..., Y_n)^T  Y ~ N_n(mu, sigma)
    mu_vec = true_mu * np.ones(n)
    Y = np.random.multivariate_normal(mu_vec, Sigma)
    
    return S, D, Y

In [16]:
# =======================================================
# 2. Log-likelihood
# =======================================================

def log_likelihood(theta, Y, D, true_mu=5):
    a, b = theta
    mu = true_mu
    
    n = len(Y)
    
    if a <= 1e-10 or b <= 1e-10:
        return -np.inf
    
    Sigma = a * np.exp(-D/b) + 1e-8 * np.eye(n)
    
    try:
        L = np.linalg.cholesky(Sigma) # Cholesky Decompose ==> |sigma| = |L^TL| =|L^T||L| = |L|^2 => 2log sigma i=0 to n 2log sigma L_ii
    except np.linalg.LinAlgError:
        return -np.inf
    
    # compute log likelihood by parts
    residual = Y - mu * np.ones(n)
    
    alpha = np.linalg.solve(L, residual) # L
    
    log_det =  2.0 * np.sum(np.log(np.diag(L)))
    
    quad_form = np.dot(alpha, alpha)
    
    return -0.5*n*np.log(2*np.pi) -0.5*log_det-0.5*quad_form
    

In [ ]:
# =======================================================
# 3.Gradient 미분정의 이용쓰
# =======================================================
def compute_gradient(theta, Y, D, eps = 1e-5):
    grad = np.zeros(3)
    for k in range(3):
        theta_plus = theta.copy()
        theta_minus = theta.copy()
        theta_plus[k] += eps
        theta_minus[k] -= eps
        grad[k] = (log_likelihood(theta_plus, Y, D) - log_likelihood(theta_minus, Y, D)) / (2 * eps) # 미분정의
    
    return grad
        

In [ ]:
# =======================================================
# 4.parameter updating
# =======================================================
def parameter_updating(theta_init, Y, D, LR, max_iter):
    theta = theta_init.copy()
    tol = 1e-6
 
    # 기록용 리스트
    #history_mu = [theta[0]]
    history_a = [theta[0]]
    history_b = [theta[1]]
    history_ll = [log_likelihood(theta, Y, D)]
 
    print("Gradient Ascent start")
    #print(f"initial: mu = {theta[0]}, a = {theta[1]}, b = {theta[2]}")
    print(f"initial:  a = {theta[0]}, b = {theta[1]}")
    print(f"initial log-likelihood : {history_ll[0]:.4f}")
    print()
 
    prev_ll = history_ll[0]
 
    for t in range(max_iter):
        grad = compute_gradient(theta, Y, D)
        theta = theta+  LR * grad
 
        current_ll = log_likelihood(theta, Y, D)
 
        # 매 iteration마다 기록
        #history_mu.append(theta[0])
        history_a.append(theta[0])
        history_b.append(theta[1])
        history_ll.append(current_ll)
 
        if (t + 1) % 200 == 0:
            # print(f"Iter {t+1:4d}: mu={theta[0]:.4f}, a={theta[1]:.4f}, "
            #       f"b={theta[2]:.4f}, loglik={current_ll:.4f}")
            print(f"Iter {t+1:4d}: a={theta[0]:.4f}, "
                  f"b={theta[1]:.4f}, loglik={current_ll:.4f}")
 
        if abs(current_ll - prev_ll) < tol:
            print(f"\nConverged! (iter {t+1})")
            break
 
        prev_ll = current_ll
 
    # return theta, history_mu, history_a, history_b, history_ll
    return theta, history_a, history_b, history_ll